In [1]:
"""Local undercoverage penalty + interval-width cost from online trajectories.

Run: python local_calibration_penalty_cost.py
Dependencies: numpy, pandas, matplotlib.

Uses EXACT centered, overlapping WINDOW=24 convention of the supplied
Helper_Scripts.calculate_metrics._iter_windows:
    even w: left=w//2-1, right=w//2
    odd w: left=right=w//2
Valid windows are entirely within the online prediction trajectory. No
online prediction is discarded from global PICP/MPIW; only the first/last
window centers lacking a full window cannot have a local metric.

For window center i:
    LPICP_i = mean(1{lower_t <= y_t <= upper_t} over window i)
    LE_i    = max(0, 1-ALPHA-LPICP_i)
    LMPIW_i = mean(width_t over window i)
    penalty_i = PRICE_UNDERCOVERAGE * LE_i
    width_cost_i = LAMBDA_WIDTH * LMPIW_i
    cost_i = penalty_i + width_cost_i

Primary COMPARABLE average cost per valid window:
    mean(cost_i) = PRICE_UNDERCOVERAGE*LPICE + LAMBDA_WIDTH*LMPIW
Also reports the user's original mixed-aggregation expression:
    PRICE_UNDERCOVERAGE*sum(LE_i) + LAMBDA_WIDTH*LMPIW
and the consistent summed expression sum(cost_i). Use the averaged
version to compare runs with different lengths.

These are ILLUSTRATIVE penalty units, NOT euro-denominated imbalance costs,
and NOT a physical energy market backtest. Window overlap intentionally
counts a single failure in multiple nearby windows.
"""

from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# -------------------------- CONFIGURATION --------------------------
TRAJECTORY_DIR = Path('./trajectories/LSTM')
OUTPUT_DIR = Path('./local_calibration_penalty_results')
METHODS = ('ACI', 'DtACI', 'AgACI', 'LACI')
OPTIONAL_METHODS = ('Weighted_ACI', 'Weighted ACI', 'WeightedACI')
DATASETS = None              # None = discover all datasets with all METHODS
ALPHA = 0.10
WINDOW = 24
PRICE_UNDERCOVERAGE = 200.0
LAMBDA_WIDTH = 0.1
# Optional experiments; each (P, lambda). Same dimensions as above.
SCENARIOS = ((50.0, 0.10), (100.0, 0.10), (200.0, 0.10),
             (200.0, 0.05), (200.0, 0.50), (200.0, 1.00))
# -------------------------------------------------------------------

COLORS = {
    'ACI': '#d84a42', 'DtACI': '#7b6ad5',
    'AgACI': '#d89a21', 'LACI': '#2382b8',
    'Weighted_ACI': '#555555', 'Weighted ACI': '#555555',
    'WeightedACI': '#555555'
}
REQUIRED_COLS = ('y_true', 'y_l_pred', 'y_u_pred')


def locate_files():
    if not TRAJECTORY_DIR.is_dir():
        raise FileNotFoundError(f'Missing trajectory directory: {TRAJECTORY_DIR}')
    found = {}
    allowed = set(METHODS) | set(OPTIONAL_METHODS)
    for path in sorted(TRAJECTORY_DIR.glob('*.csv')):
        if '__' not in path.stem:
            continue
        dataset, method = path.stem.rsplit('__', 1)
        if method not in allowed or (DATASETS is not None and dataset not in DATASETS):
            continue
        if method in found.setdefault(dataset, {}):
            raise ValueError(f'Duplicate trajectory method for {dataset}: {method}')
        found[dataset][method] = path
    if not found:
        raise FileNotFoundError(f'No matching <dataset>__<method>.csv in {TRAJECTORY_DIR}')
    return found


def read_data(path):
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLS if col not in df]
    if missing:
        raise ValueError(f'{path}: missing required columns {missing}')
    for col in REQUIRED_COLS:
        df[col] = pd.to_numeric(df[col], errors='raise')
    if not np.isfinite(df['y_true'].to_numpy(dtype=float)).all():
        raise ValueError(f'{path}: y_true must be finite at every point')
    lo = df['y_l_pred'].to_numpy(dtype=float)
    hi = df['y_u_pred'].to_numpy(dtype=float)
    if np.isnan(lo).any() or np.isnan(hi).any():
        raise ValueError(f'{path}: interval endpoints contain NaN; no rows skipped')
    if not len(df):
        raise ValueError(f'{path}: empty trajectory')
    if 'step' in df and not df['step'].is_unique:
        raise ValueError(f'{path}: duplicate step labels')
    return df


def load_trajectories(paths):
    missing = [m for m in METHODS if m not in paths]
    if missing:
        raise ValueError(f'Dataset missing methods {missing}; provided: {sorted(paths)}')
    used = list(METHODS) + [m for m in OPTIONAL_METHODS if m in paths]
    frames = {method: read_data(paths[method]) for method in used}
    reference_name = METHODS[0]
    ref = frames[reference_name]
    ref_true = ref['y_true'].to_numpy(float)
    # for method, df in frames.items():
    #     if len(df) != len(ref):
    #         raise ValueError(f'Unaligned row counts: {reference_name}={len(ref)}, {method}={len(df)}')
    #     if ('step' in ref) and ('step' in df):
    #         if not np.array_equal(ref['step'].to_numpy(), df['step'].to_numpy()):
    #             raise ValueError(f'{method}: step values do not match {reference_name}')
    #     if not np.allclose(df['y_true'].to_numpy(float), ref_true,
    #                        atol=1e-4, rtol=1e-6):
    #         raise ValueError(f'{method}: y_true values disagree with {reference_name}; '
    #                          'check alignment, scaling and evaluation period')
    return frames


def centered_windows(n, w):
    if not isinstance(w, int) or w < 1:
        raise ValueError('WINDOW must be a positive integer')
    left = w // 2 - 1 if w % 2 == 0 else w // 2
    right = w // 2
    for center in range(n):
        start, end = center - left, center + right  # inclusive end
        if start >= 0 and end < n:
            yield center, start, end


def calc_method_rows(dataset, method, df):
    y = df['y_true'].to_numpy(dtype=float)
    lo = df['y_l_pred'].to_numpy(dtype=float)
    hi = df['y_u_pred'].to_numpy(dtype=float)
    covered = ((lo <= y) & (y <= hi)).astype(float)
    # Finite crossed bounds represent empty sets: uncovered with zero width.
    # Mathematical full-line bounds represent certain coverage but infinite width.
    with np.errstate(invalid='ignore'):
        width = np.where(lo > hi, 0.0, hi - lo)
    if np.isnan(width).any():
        raise ValueError(f'{dataset}/{method}: undefined interval width')
    rows = []
    for center, start, end in centered_windows(len(df), WINDOW):
        local_coverage = float(covered[start:end + 1].mean())
        local_error = max(1.0 - ALPHA - local_coverage, 0.0)
        local_width = float(width[start:end + 1].mean())
        penalty = PRICE_UNDERCOVERAGE * local_error
        width_cost = LAMBDA_WIDTH * local_width
        rows.append({
            'dataset': dataset, 'method': method,
            'center_row': center, 'start_row': start,
            'end_row_inclusive': end,
            'LPICP_i': local_coverage, 'LE_i': local_error,
            'LPIA_i': int(local_coverage >= 1.0 - ALPHA - 1e-12),
            'LMPIW_i': local_width,
            'penalty_i': penalty, 'width_cost_i': width_cost,
            'cost_i': penalty + width_cost,
        })
    if not rows:
        raise ValueError(f'{dataset}/{method}: {len(df)} predictions < WINDOW={WINDOW}')
    per_window = pd.DataFrame(rows)
    lpice = float(per_window['LE_i'].mean())
    lmpiw = float(per_window['LMPIW_i'].mean())
    n = len(per_window)
    summary = {
        'dataset': dataset, 'method': method,
        'n_predictions': len(df), 'n_local_windows': n,
        'PICP': float(covered.mean()),
        'MPIW': float(width.mean()),
        'MLPICP': float(per_window['LPICP_i'].mean()),
        'LPICE': lpice,
        'LPIA': float(per_window['LPIA_i'].mean()),
        'LMPIW': lmpiw,
        'sum_LE': float(per_window['LE_i'].sum()),
        'price_undercoverage': PRICE_UNDERCOVERAGE,
        'lambda_width': LAMBDA_WIDTH,
        'total_penalty_all_windows': float(per_window['penalty_i'].sum()),
        'total_width_cost_all_windows': float(per_window['width_cost_i'].sum()),
        'total_cost_sum_windows': float(per_window['cost_i'].sum()),
        'mean_cost_per_window': float(per_window['cost_i'].mean()),
        # User's literal proposal mixes a sum and a mean. Report for transparency.
        'cost_user_proposed_P_sumLE_plus_lambda_LMPIW':
            PRICE_UNDERCOVERAGE * float(per_window['LE_i'].sum()) +
            LAMBDA_WIDTH * lmpiw,
        'n_finite_crossed': int(((lo > hi) & np.isfinite(lo) & np.isfinite(hi)).sum()),
        'fraction_infinite_intervals': float((~np.isfinite(lo) | ~np.isfinite(hi)).mean()),
    }
    # Total cost is the unnormalized sum across all valid overlapping windows.
    if np.isfinite(summary['total_cost_sum_windows']):
        assert np.isclose(summary['total_cost_sum_windows'],
                          n * summary['mean_cost_per_window'], atol=1e-7)
    # Algebraic checks on cost definition; no data exclusion.
    if np.isfinite(summary['mean_cost_per_window']):
        expected = PRICE_UNDERCOVERAGE * lpice + LAMBDA_WIDTH * lmpiw
        assert np.isclose(summary['mean_cost_per_window'], expected, atol=1e-8)
    return per_window, summary


def sensitivity_rows(summary):
    result = []
    scenarios = tuple(dict.fromkeys(((PRICE_UNDERCOVERAGE, LAMBDA_WIDTH),) + SCENARIOS))
    for price, width_lambda in scenarios:
        if price < 0 or width_lambda < 0:
            raise ValueError('Cost coefficients must be nonnegative')
        result.append({
            'dataset': summary['dataset'], 'method': summary['method'],
            'price_undercoverage': price, 'lambda_width': width_lambda,
            'mean_cost_per_window': price * summary['LPICE'] + width_lambda * summary['LMPIW'],
            'total_cost_sum_windows': summary['n_local_windows'] *
                                      (price * summary['LPICE'] + width_lambda * summary['LMPIW']),
        })
    return result


def plot_dataset(dataset, detail, summary):
    safe = re.sub(r'[^a-zA-Z0-9_-]', '_', dataset)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for method in summary['method']:
        d = detail.loc[detail.method == method]
        axes[0].plot(d['center_row'], d['LE_i'], label=method,
                     color=COLORS.get(method), lw=1.3, alpha=0.85)
    axes[0].set_xlabel('Online prediction index (window center)')
    axes[0].set_ylabel('Local undercoverage error $LE_i$')
    axes[0].set_title('Overlapping centered 24-point windows')
    axes[0].legend(frameon=False, fontsize=8)
    ordered = summary.sort_values('mean_cost_per_window')
    axes[1].bar(ordered['method'], ordered['mean_cost_per_window'],
                color=[COLORS.get(m, '#888888') for m in ordered['method']])
    axes[1].set_ylabel('Mean composite cost per window (arbitrary units)')
    axes[1].set_title(f'P={PRICE_UNDERCOVERAGE:g}, lambda={LAMBDA_WIDTH:g}')
    axes[1].tick_params(axis='x', rotation=15)
    fig.suptitle(f'{dataset} — calibration penalty + interval width')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{safe}_calibration_penalty_cost.png', dpi=175)
    plt.close(fig)


def main():
    if not 0 < ALPHA < 1:
        raise ValueError('ALPHA must lie strictly between 0 and 1')
    if PRICE_UNDERCOVERAGE < 0 or LAMBDA_WIDTH < 0:
        raise ValueError('Price and lambda must be nonnegative')
    files = locate_files()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    details, summaries, sensitivities = [], [], []
    for dataset, paths in sorted(files.items()):
        frames = load_trajectories(paths)
        dataset_details, dataset_summaries = [], []
        for method, df in frames.items():
            d, s = calc_method_rows(dataset, method, df)
            dataset_details.append(d)
            dataset_summaries.append(s)
            sensitivities.extend(sensitivity_rows(s))
        detail = pd.concat(dataset_details, ignore_index=True)
        summary = pd.DataFrame(dataset_summaries)
        baseline = summary.loc[summary.method == 'ACI', 'mean_cost_per_window'].iloc[0]
        summary['saving_vs_ACI_pct'] = (
            100 * (baseline - summary['mean_cost_per_window']) / baseline
            if baseline > 0 and np.isfinite(baseline) else np.nan
        )
        details.append(detail)
        summaries.append(summary)
        plot_dataset(dataset, detail, summary)
        print(f'\n{dataset} | {len(frames)} methods | '
              f'{len(next(iter(frames.values())))} predictions | '
              f'{len(detail) // len(frames)} valid centered windows')
        print(summary[['method', 'LPICE', 'LPIA', 'LMPIW', 'PICP', 'MPIW',
                       'mean_cost_per_window', 'total_cost_sum_windows',
                        'saving_vs_ACI_pct']]
              .to_string(index=False, float_format=lambda v: f'{v:.5f}'))
    all_details = pd.concat(details, ignore_index=True)
    all_summaries = pd.concat(summaries, ignore_index=True)
    scenario_df = pd.DataFrame(sensitivities)
    # Savings computed independently for every dataset+cost scenario.
    baseline = scenario_df.loc[scenario_df.method == 'ACI',
                               ['dataset', 'price_undercoverage', 'lambda_width',
                                'mean_cost_per_window']].rename(columns={
                                    'mean_cost_per_window': 'ACI_mean_cost_per_window'})
    scenario_df = scenario_df.merge(
        baseline, on=['dataset', 'price_undercoverage', 'lambda_width'],
        validate='many_to_one')
    with np.errstate(divide='ignore', invalid='ignore'):
        scenario_df['saving_vs_ACI_pct'] = np.where(
            scenario_df['ACI_mean_cost_per_window'] > 0,
            100 * (scenario_df['ACI_mean_cost_per_window'] -
                   scenario_df['mean_cost_per_window']) /
                  scenario_df['ACI_mean_cost_per_window'], np.nan)
    all_details.to_csv(OUTPUT_DIR / 'local_window_costs.csv', index=False)
    all_summaries.to_csv(OUTPUT_DIR / 'main_paper_table.csv', index=False)
    scenario_df.to_csv(OUTPUT_DIR / 'cost_sensitivity.csv', index=False)
    print(f'\nSaved all results to: {OUTPUT_DIR.resolve()}')
    print('IMPORTANT: costs are illustrative calibration-efficiency scores, NOT EUR or reBAP.')
    print('WINDOW=24 is measured in trajectory rows, not necessarily hours.')
    print('Each local window overlaps its neighbors (paper-style centered windows).')


if __name__ == '__main__':
    main()



50Hertz | 4 methods | 4480 predictions | 4457 valid centered windows
method   LPICE    LPIA    LMPIW    PICP     MPIW  mean_cost_per_window  total_cost_sum_windows  saving_vs_ACI_pct
   ACI 0.03977 0.67085 49.14174 0.90469 49.15701              12.86833             57354.13824            0.00000
 DtACI 0.03887 0.62688 48.10297 0.90201 48.10907              12.58421             56087.82690            2.20788
 AgACI 0.03648 0.62643 49.63582 0.90379 49.64069              12.25922             54639.35262            4.73337
  LACI 0.02636 0.73480 55.12378 0.92946 55.12192              10.78349             48062.00004           16.20134

Amprion | 4 methods | 4480 predictions | 4457 valid centered windows
method   LPICE    LPIA    LMPIW    PICP     MPIW  mean_cost_per_window  total_cost_sum_windows  saving_vs_ACI_pct
   ACI 0.04201 0.67848 32.06751 0.90737 32.07981              11.60889             51740.82297            0.00000
 DtACI 0.03937 0.66053 30.89211 0.90357 30.90093              